# Zero-Shot Classification - Hugging Face (Advanced Optimization)

Notebook ini menggunakan model `facebook/bart-large-mnli` dengan pendekatan **Confidence Threshold** untuk meminimalkan *false positive* (teks biasa yang dikira promosi).

In [ ]:
# Install dependencies (menggunakan versi CPU untuk stabilitas di Windows)
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cpu
!pip install pandas transformers==4.38.2 scikit-learn tqdm

In [ ]:
import os
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# Inisialisasi model
model_name = "facebook/bart-large-mnli"
print(f"Memuat model: {model_name}...")

try:
    classifier = pipeline(
        "zero-shot-classification", 
        model=model_name,
        device=-1 # CPU mode
    )
    print("Model berhasil dimuat.")
except Exception as e:
    print(f"Gagal memuat model utama, mencoba model alternatif: {e}")
    classifier = pipeline(
        "zero-shot-classification", 
        model="valhalla/distilbart-mnli-12-3",
        device=-1
    )
    print("Model alternatif berhasil dimuat.")

def classify_text_advanced(text, threshold=0.65):
    """
    Fungsi klasifikasi dengan Confidence Threshold.
    Hanya menganggap PROMOSI jika keyakinan model di atas threshold.
    """
    if pd.isna(text) or str(text).strip() == "":
        return "BUKAN"
    
    try:
        # Hypothesis yang lebih fokus pada topik
        hypothesis = "This text is about {}"
        
        # Kita pakai label bahasa inggris sementara agar BART (yang dilatih bahasa inggris) paham lebih baik,
        # namun tetap memberikan teks asli ke model.
        candidate_labels = ["online gambling promotion or advertisement", "general conversation, news, or complaints"]
        
        result = classifier(text, candidate_labels=candidate_labels, hypothesis_template=hypothesis)
        
        # Ambil skor dan label terbaik
        top_label = result['labels'][0]
        top_score = result['scores'][0]
        
        # Logika Threshold: Jika model ragu atau label terbaik bukan promosi, maka BUKAN
        if top_label == "online gambling promotion or advertisement" and top_score >= threshold:
            return "PROMOSI"
        else:
            return "BUKAN"
            
    except Exception as e:
        return f"ERROR: {str(e)}"

## Pemrosesan Data

Ubah nilai `threshold` (0.0 sampai 1.0) di bawah ini jika ingin lebih ketat atau lebih longgar.
Semakin tinggi nilai threshold, semakin sedikit teks yang akan ditandai sebagai PROMOSI (lebih ketat).

In [ ]:
input_path = r"C:\Users\mahesa\Downloads\evaluasi_30.csv"
output_path = "zero_shot_results_final.csv"
CONFIDENCE_THRESHOLD = 0.65 # Silakan sesuaikan (misal 0.7 untuk lebih ketat)

if os.path.exists(input_path):
    df = pd.read_csv(input_path)
    
    if 'tweet_text' in df.columns:
        print(f"Memproses {len(df)} baris data dengan threshold {CONFIDENCE_THRESHOLD}...")
        tqdm.pandas()
        
        df['predicted_label'] = df['tweet_text'].progress_apply(lambda x: classify_text_advanced(x, threshold=CONFIDENCE_THRESHOLD))
        
        df.to_csv(output_path, index=False)
        print(f"Hasil disimpan ke: {output_path}")
        
        # Tampilkan beberapa contoh hasil klasifikasi
        print("\nContoh Hasil:")
        print(df[['tweet_text', 'predicted_label']].head(15))
    else:
        print("Kolom 'tweet_text' tidak ditemukan dalam file.")
else:
    print(f"File tidak ditemukan: {input_path}")

## Evaluasi Akurasi

In [ ]:
target_column = 'label' if 'label' in df.columns else 'label_llm' if 'label_llm' in df.columns else None

if target_column and 'predicted_label' in df.columns:
    mask = ~df['predicted_label'].str.contains("ERROR", na=False)
    valid_df = df[mask].copy()
    
    y_true = valid_df[target_column].astype(str).str.strip()
    y_pred = valid_df['predicted_label'].astype(str).str.strip()
    
    print(f"Target Column: {target_column}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy: {accuracy * 100:.2f}%")
else:
    print("Kolom target tidak ditemukan untuk perhitungan akurasi.")